# Extended Data Figure 6: Depth plane

Colour shows the mean long-run cooperation $\bar\rho$ over 512 area-uniform initial strategy compositions for
every integer pair $0\le k_1,k_2\le 7$ with full readout, at $b/c=5$ and $e_1=e_2=0.02$. Dots mark pairs at which a
pure DISC population excludes rare ALLD and is not invaded by rare ALLC. Run from this directory. Saved data are
used by default; `RECOMPUTE=True` regenerates all 256 conditions with the verified engine in `../engine`.

## Configuration

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle

ROOT = Path.cwd()
ENGINE = (ROOT / '..' / 'engine').resolve()
sys.path.insert(0, str(ENGINE))
DATA = ROOT / 'data' / 'extended_data_figure6'
FIGURES = ROOT / 'figures' / 'extended_data_figure6'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
RECOMPUTE = False

NORM_ORDER = ('SC', 'SH', 'SJ', 'SS')
NORM_NAMES = {'SC': 'Scoring', 'SH': 'Shunning', 'SJ': 'Stern Judging', 'SS': 'Simple Standing'}
B_OVER_C = 5.0
E1 = E2 = 0.02
SEED = 2026
N_INITIAL = 512
DEPTH_VALUES = tuple(range(8))

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.linewidth': .7, 'xtick.direction': 'out',
                     'ytick.direction': 'out', 'hatch.linewidth': .45, 'pdf.fonttype': 42, 'svg.fonttype': 'none'})

## Numerical method

Identical to Figure 4 (see `Figure4_Total_Depth.ipynb` and `../engine/README.md`): all-good-selected reputations,
exact evaluation wherever the reputation map is not smooth, midpoint integration to convergence, and attractor-based
outcome classes. Local protection uses the pure-DISC resident reputation reached from all-good, refined to a residual
below $10^{-13}$, and the rule of Extended Data Figures 3 and 5: $\lambda_{\rm ALLD}<-10^{-10}$ and
$\lambda_{\rm ALLC}\le10^{-10}$.

In [2]:
import irsim, analysis

def compute_condition(norm, k1, k2):
    p = irsim.params(norm, k1, k2, 1.0, 1.0, E1, E2, B_OVER_C, 1.0)
    out, info = irsim.run(p, irsim.halton_shifted(N_INITIAL, SEED), bs=(B_OVER_C,))
    o = out[0]; rho = (1 - E1) * o[:, 6]
    row = dict(norm=norm, k1=k1, k2=k2, K=k1 + k2, alpha=1.0, beta=1.0, b_over_c=B_OVER_C, e1=E1, e2=E2,
               sample='halton2026', **analysis.summarize_outcomes(rho),
               max_steps=int(o[:, 7].max()), n_step_cap=int((o[:, 7] >= 400 * 500).sum()),
               max_final_strategy_residual=float(o[:, 8].max()), lattice_max_residual=info[0],
               flagged_cells=int(info[1]), exact_evaluations=int(info[2]))
    inv = analysis.pure_disc_invasion(norm, k1, k2, 1.0, 1.0, B_OVER_C, E1, E2)
    row.update(lambda_ALLD=inv['lambda_ALLD'], lambda_ALLC=inv['lambda_ALLC'],
               disc_locally_protected=bool(analysis.locally_protected(inv['lambda_ALLD'], inv['lambda_ALLC'])))
    return row, o

## Data

In [3]:
path = DATA / 'depth_plane_results.csv'
if RECOMPUTE:
    rows, finals = [], []
    for norm in NORM_ORDER:
        for k2 in DEPTH_VALUES:
            for k1 in DEPTH_VALUES:
                row, o = compute_condition(norm, k1, k2); rows.append(row); finals.append(o)
                print(f"{norm} ({k1},{k2}): mean cooperation {row['mean_cooperation']:.4f}", flush=True)
    data = pd.DataFrame(rows); data.to_csv(path, index=False)
    np.savez_compressed(DATA / 'depth_plane_results_final_states.npz',
                        final_frequencies=np.array([o[:, :3] for o in finals], np.float32),
                        final_cooperation=np.array([(1 - E1) * o[:, 6] for o in finals], np.float32),
                        norm=data.norm.to_numpy(), k1=data.k1.to_numpy(), k2=data.k2.to_numpy())
data = pd.read_csv(path)
assert len(data) == 4 * 8 ** 2 and not data.duplicated(['norm', 'k1', 'k2']).any()
for norm, part in data.groupby('norm'):
    assert set(zip(part.k1, part.k2)) == {(i, j) for i in DEPTH_VALUES for j in DEPTH_VALUES}
assert np.allclose(data[['b_over_c', 'e1', 'e2', 'alpha', 'beta']], [B_OVER_C, E1, E2, 1., 1.])
assert data.outcome_gap_ok.all() and data.mean_cooperation.between(0, 1).all()
np.testing.assert_array_equal(data.disc_locally_protected.astype(bool),
                              analysis.locally_protected(data.lambda_ALLD, data.lambda_ALLC))
summary = data.groupby('norm', sort=False).agg(conditions=('mean_cooperation', 'size'),
    min_mean_cooperation=('mean_cooperation', 'min'), max_mean_cooperation=('mean_cooperation', 'max'),
    protected_cells=('disc_locally_protected', 'sum'), max_final_strategy_residual=('max_final_strategy_residual', 'max'))
summary.to_csv(DATA / 'grid_summary.csv')
display(summary)

      conditions  min_mean_cooperation  max_mean_cooperation  protected_cells  max_final_strategy_residual
norm                                                                                                      
SC            64          1.023733e-12              0.888125               50                     0.000015
SH            64          3.581030e-12              0.813477               50                     0.000015
SJ            64          0.000000e+00              0.742656               50                     0.000015
SS            64          7.433759e-02              0.835961               50                     0.000015


## Panels

In [4]:
CMAP = LinearSegmentedColormap.from_list('cooperation_blue', ['#F4F8FB', '#DCEAF3', '#A9CFE3', '#6FA8C9', '#2F6F9F'])
HATCH_COLOR = (.1, .1, .1, .32)

def arrays(norm):
    part = data.loc[data.norm.eq(norm)]
    grid = lambda column: part.pivot(index='k2', columns='k1', values=column).loc[list(DEPTH_VALUES), list(DEPTH_VALUES)].to_numpy()
    return grid('mean_cooperation'), grid('disc_locally_protected').astype(bool)

def draw_panel(ax, norm, panel=None):
    cooperation, protected = arrays(norm)
    edges = np.arange(-.5, 8., 1.)
    image = ax.pcolormesh(edges, edges, cooperation, cmap=CMAP, vmin=0, vmax=1, shading='flat', rasterized=True)
    for j, i in np.argwhere(protected):
        ax.add_patch(Rectangle((i - .5, j - .5), 1, 1, facecolor='none', edgecolor=HATCH_COLOR, linewidth=0, hatch='....'))
    ax.set(xlim=(-.5, 7.5), ylim=(-.5, 7.5), xticks=DEPTH_VALUES, yticks=DEPTH_VALUES,
           xlabel=r'Tolerance depth $k_1$', ylabel=r'Strictness depth $k_2$', title=NORM_NAMES[norm], aspect='equal')
    ax.tick_params(length=2.5, pad=2)
    if panel: ax.text(-.16, 1.04, panel, transform=ax.transAxes, fontweight='bold', fontsize=11)
    return image

def save_figure(fig, name):
    for extension in ('pdf', 'svg'):
        fig.savefig(FIGURES / f'{name}.{extension}', bbox_inches='tight', pad_inches=.04)

HATCH_LEGEND = Patch(facecolor='white', edgecolor='#777777', hatch='....', label='Pure DISC resists rare ALLD and ALLC')

In [5]:
for norm in NORM_ORDER:
    fig, ax = plt.subplots(figsize=(4.0, 3.8))
    fig.subplots_adjust(left=.15, right=.80, bottom=.23, top=.90)
    image = draw_panel(ax, norm)
    fig.colorbar(image, cax=fig.add_axes([.84, .25, .035, .59]), ticks=[0, .5, 1], label=r'Mean cooperation $\bar\rho$')
    fig.legend(handles=[HATCH_LEGEND], loc='lower center', bbox_to_anchor=(.50, .025), frameon=False, fontsize=8)
    save_figure(fig, f'panel_{norm}')
    plt.close(fig)

fig, axes = plt.subplots(1, 4, figsize=(12.6, 3.8), sharex=True, sharey=True)
fig.subplots_adjust(left=.065, right=.91, bottom=.25, top=.86, wspace=.18)
for i, (ax, norm) in enumerate(zip(axes, NORM_ORDER)):
    image = draw_panel(ax, norm, 'abcd'[i])
    if i: ax.set_ylabel('')
fig.colorbar(image, cax=fig.add_axes([.932, .28, .014, .54]), ticks=[0, .5, 1], label=r'Mean cooperation $\bar\rho$')
fig.legend(handles=[HATCH_LEGEND], loc='lower center', bbox_to_anchor=(.49, .055), frameon=False, fontsize=9)
save_figure(fig, 'Extended_Data_Figure6_combined')
plt.show(); plt.close(fig)
assert len(list(FIGURES.glob('*.pdf'))) == len(list(FIGURES.glob('*.svg'))) == 5
print('Saved four panels and one combined figure in PDF and SVG.')

Saved four panels and one combined figure in PDF and SVG.
